# LeakAudit - 08 - Mitigation and conformal abstention (E10, E11)

**Two constructive experiments on the hardest task, outdoor vs indoor.**

- **E10** Try standard shortcut-mitigation methods (ERM, an approximate Group-DRO reweighting, and JTT). If none beats the baseline, the confound lives in data collection, not in the algorithm.
- **E11** Split / Mondrian conformal prediction gives guaranteed coverage and a principled abstention region: the honest way to say 'not decidable here'.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation
cfg.apply_mpl_style()
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

X1, X1_max, M1, HZ = io_utils.load_d1()
y5 = M1.cls.to_numpy(); site = M1.site.to_numpy()
mask = np.isin(y5, ["out", "in"])
Xoi, yoi, soi = X1[mask], y5[mask], site[mask]
print(f"outdoor vs indoor: {len(yoi):,} rows, "
      f"{(yoi=='out').sum()} out / {(yoi=='in').sum()} in")

outdoor vs indoor: 34,663 rows, 19731 out / 14932 in


## E10 - shortcut mitigation attempts

Group-DRO is approximated by up-weighting sites with high training error; JTT
up-weights points the first pass misclassifies. Both need only the site label,
which we have.

In [3]:
def group_dro_weights(Xtr, ytr, gtr, seed):
    m0 = evaluation.make_model("gbdt", seed).fit(Xtr, ytr)
    err = pd.DataFrame({"g": gtr, "e": (m0.predict(Xtr) != ytr).astype(float)}) \
            .groupby("g").e.mean()
    w = pd.Series(gtr).map(err).to_numpy()
    return (w + 0.1) / (w + 0.1).mean()

def jtt_weights(Xtr, ytr, seed, upweight=5.0):
    m0 = evaluation.make_model("gbdt", seed).fit(Xtr, ytr)
    w = np.ones(len(ytr)); w[m0.predict(Xtr) != ytr] = upweight
    return w

rows = []
for seed in range(min(cfg.N_SEEDS, 5)):
    a, b = next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                .split(np.arange(len(yoi)), yoi, groups=soi))
    if len(a) > 20000:
        a = np.random.RandomState(seed).choice(a, 20000, replace=False)
    if len(np.unique(yoi[a])) < 2 or len(np.unique(yoi[b])) < 2:
        continue
    rows.append(dict(method="ERM", seed=seed,
                     **evaluation.evaluate(Xoi[a], yoi[a], Xoi[b], yoi[b], "gbdt", seed)))
    for nm, wf in [("GroupDRO_approx", lambda: group_dro_weights(Xoi[a], yoi[a], soi[a], seed)),
                   ("JTT", lambda: jtt_weights(Xoi[a], yoi[a], seed))]:
        w = wf()
        m = evaluation.make_model("gbdt", seed).fit(Xoi[a], yoi[a], sample_weight=w)
        p = m.predict(Xoi[b])
        rows.append(dict(method=nm, seed=seed,
                         acc=accuracy_score(yoi[b], p),
                         bacc=balanced_accuracy_score(yoi[b], p),
                         mf1=f1_score(yoi[b], p, average="macro"),
                         base=pd.Series(yoi[b]).value_counts(normalize=True).max()))

e10 = pd.DataFrame(rows)
s10 = e10.groupby("method").agg(acc=("acc", "mean"), sd=("acc", "std"),
                                base=("base", "mean"), n=("acc", "size")).reset_index()
print(s10.round(4).to_string(index=False))
e10.to_csv(cfg.RESULTS / "E10_mitigation_raw.csv", index=False)
s10.to_csv(cfg.RESULTS / "E10_mitigation_summary.csv", index=False)
if s10.acc.max() <= s10[s10.method == "ERM"].acc.iloc[0] + 0.01:
    print("\n-> no method beats ERM: the confound is in data collection, not the algorithm.")

         method    acc     sd   base  n
            ERM 0.5016 0.0259 0.5601  5
GroupDRO_approx 0.5061 0.0210 0.5601  5
            JTT 0.5114 0.0224 0.5601  5

-> no method beats ERM: the confound is in data collection, not the algorithm.


## E11 - conformal abstention

Guaranteed coverage of 1 - alpha. A multi-class prediction set is an
abstention. High abstention at low error is the honest signal that the task is
not decidable from a single-site spectrum.

In [4]:
rows = []
for seed in range(min(cfg.N_SEEDS, 5)):
    a, b = next(GroupShuffleSplit(1, test_size=.4, random_state=seed)
                .split(np.arange(len(yoi)), yoi, groups=soi))
    if len(np.unique(yoi[a])) < 2 or len(np.unique(yoi[b])) < 2:
        continue
    cal, te = train_test_split(b, test_size=.5, random_state=seed, stratify=yoi[b])
    if len(a) > 20000:
        a = np.random.RandomState(seed).choice(a, 20000, replace=False)
    model = evaluation.make_model("gbdt", seed).fit(Xoi[a], yoi[a])
    cls = model.classes_
    for alpha in [0.05, 0.10, 0.20]:
        for mond in [False, True]:
            S, _ = evaluation.split_conformal(model, Xoi[cal], yoi[cal], Xoi[te],
                                              alpha=alpha, classes=cls, mondrian=mond)
            rep = evaluation.conformal_report(S, yoi[te], cls)
            rows.append(dict(seed=seed, alpha=alpha, mondrian=mond, target=1 - alpha, **rep))

e11 = pd.DataFrame(rows)
s11 = e11.groupby(["alpha", "mondrian"]).agg(
        coverage=("coverage", "mean"), target=("target", "mean"),
        set_size=("mean_set_size", "mean"), abstain=("abstain_rate", "mean"),
        acc_decided=("acc_on_decided", "mean"), n=("coverage", "size")).reset_index()
print(s11.round(4).to_string(index=False))
e11.to_csv(cfg.RESULTS / "E11_conformal_raw.csv", index=False)
s11.to_csv(cfg.RESULTS / "E11_conformal_summary.csv", index=False)
print("\n-> coverage tracks the target; a high abstention rate is the honest verdict.")

 alpha  mondrian  coverage  target  set_size  abstain  acc_decided  n
  0.05     False    0.9507    0.95    1.8835   0.8835       0.5773  5
  0.05      True    0.9498    0.95    1.8847   0.8847       0.5648  5
  0.10     False    0.9019    0.90    1.7840   0.7840       0.5459  5
  0.10      True    0.9014    0.90    1.7815   0.7815       0.5480  5
  0.20     False    0.8016    0.80    1.5821   0.5821       0.5247  5
  0.20      True    0.7980    0.80    1.5676   0.5676       0.5325  5

-> coverage tracks the target; a high abstention rate is the honest verdict.


## E11b - selective risk control (an error-rate guarantee)

Conformal coverage is **not** an accuracy guarantee: it states only that the
prediction *set* contains the truth with probability 1 - alpha. A utility
acting on alarms needs something different - a bound on how often an accepted
automated decision is wrong.

We therefore also control the *joint* risk

    P(prediction wrong AND accepted) <= alpha

which is monotone in the confidence threshold and admits the conformal risk
control correction of Angelopoulos et al. (2024). Read operationally: 'of every
100 alarms, at most alpha x 100 receive a confidently wrong automated
decision.' Everything below the threshold is referred for manual verification.

**What this does NOT say.** Because P(wrong AND accepted) = P(wrong | accepted)
x P(accepted), a joint risk below alpha does not bound the error rate *within*
the accepted subset. At alpha = 0.05 the joint risk is about 5 % of all alarms
while roughly 42 % of the accepted decisions are wrong, because only about 12 %
of alarms are accepted at all. Any claim of the form 'automated decisions are
at least (1 - alpha) accurate' would be false here; controlling the conditional
error rate requires a selective-risk formulation, which we do not claim.

Two calibration regimes are compared, because the exchangeability assumption
behind the bound is itself at stake in this paper:

* **row-level** - every record treated as exchangeable (the textbook setting);
* **site-clustered** - calibration on per-site mean losses, which respects the
  clustering that field data actually has.

We report the realised risk and the number of runs in which it exceeded the
budget, rather than asserting the bound holds. We also report the realised
selective risk on the accepted subset alongside its majority-class baseline,
because that comparison - not the guarantee itself - tells us whether
automation is worth anything here.

In [5]:
rows = []
for seed in range(min(cfg.N_SEEDS, 5)):
    a, b = next(GroupShuffleSplit(1, test_size=.4, random_state=seed)
                .split(np.arange(len(yoi)), yoi, groups=soi))
    if len(np.unique(yoi[a])) < 2 or len(np.unique(yoi[b])) < 2:
        continue
    cal, te = train_test_split(b, test_size=.5, random_state=seed, stratify=yoi[b])
    if len(a) > 20000:
        a = np.random.RandomState(seed).choice(a, 20000, replace=False)
    model = evaluation.make_model("gbdt", seed).fit(Xoi[a], yoi[a])
    for alpha in [0.01, 0.05, 0.10, 0.20]:
        # Two calibration regimes. 'row' treats every record as exchangeable,
        # which is the textbook setting; 'site' calibrates on per-site mean
        # losses, which respects the clustering present in field data.
        for mode, g in [("row", None), ("site", soi[cal])]:
            r = evaluation.selective_risk_control(model, Xoi[cal], yoi[cal],
                                                  Xoi[te], yoi[te], alpha=alpha,
                                                  groups_cal=g)
            r["mode"] = mode
            r["seed"] = seed
            rows.append(r)

e11b = pd.DataFrame(rows)
s11b = e11b.groupby(["mode", "alpha"]).agg(
        threshold=("threshold", "mean"),
        accept_rate=("accept_rate", "mean"),
        joint_risk=("joint_risk", "mean"),
        violations=("risk_exceeds_alpha", "sum"),
        selective_risk=("selective_risk", "mean"),
        acc_accepted=("acc_on_accepted", "mean"),
        baseline_accepted=("baseline_on_accepted", "mean"),
        n=("threshold", "size")).reset_index()
print(s11b.round(4).to_string(index=False))
e11b.to_csv(cfg.RESULTS / "E11b_selective_risk_raw.csv", index=False)
s11b.to_csv(cfg.RESULTS / "E11b_selective_risk_summary.csv", index=False)

print("\n--- reading the table ---")
for mode in s11b["mode"].unique():
    sub = s11b[s11b["mode"] == mode]
    v = int(sub.violations.sum()); tot = int(sub.n.sum())
    print(f"  {mode:5s} calibration: realised risk exceeded alpha in "
          f"{v}/{tot} runs")
print()
print("-> Row-level calibration treats every record as exchangeable. Field")
print("   records are clustered by site, so that assumption is optimistic and")
print("   the realised risk overshoots alpha by a small but systematic margin.")
print("   Calibrating on per-site mean losses restores validity at the cost of")
print("   a far lower acceptance rate.")
print("-> Either way, accepted decisions barely exceed - and at looser levels")
print("   fall below - the majority-class baseline, so this task is not")
print("   automatable from a single-site spectrum however the threshold is set.")

mode  alpha  threshold  accept_rate  joint_risk  violations  selective_risk  acc_accepted  baseline_accepted  n
 row   0.01     0.9874       0.0314      0.0107           3          0.3414        0.6586             0.6109  5
 row   0.05     0.9650       0.1163      0.0493           3          0.4228        0.5772             0.5692  5
 row   0.10     0.9334       0.2160      0.0981           2          0.4541        0.5459             0.5544  5
 row   0.20     0.8499       0.4178      0.1983           1          0.4751        0.5249             0.5387  5
site   0.01     1.0000       0.0000      0.0000           0             NaN           NaN                NaN  5
site   0.05     0.9971       0.0048      0.0011           0          0.1889        0.8111             0.7688  5
site   0.10     0.9499       0.1581      0.0692           1          0.4360        0.5640             0.5662  5
site   0.20     0.8584       0.3930      0.1850           2          0.4721        0.5279             0.